In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
transformers \
accelerate \
sentence-transformers \
faiss-cpu \
chromadb \
langchain-groq \
langchain-openrouter \
gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/13

In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/ufc-rag-proj')


Mounted at /content/drive


In [4]:
import duckdb
from pathlib import Path
from langchain.tools import tool
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from typing import Optional

PROJECT_ROOT = Path('/content/drive/MyDrive/ufc-rag-proj')
DB_PATH = PROJECT_ROOT / "data/ufc.duckdb"
CHROMA_PATH = PROJECT_ROOT / "vectorstore" / "chroma_db"

/tmp/ipykernel_5303/2212478063.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [5]:
conn = duckdb.connect(DB_PATH)
conn.execute("SHOW TABLES").fetchdf()

,name
0,events
1,fight_performances
2,fighters
3,fights


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
vectorstore = Chroma(
    persist_directory=str(CHROMA_PATH),
    embedding_function=embedding_model
)

print(vectorstore._collection.count())

/tmp/ipykernel_5303/1431764167.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


12553


In [8]:
@tool
def retrieve_documents(query: str, k: int = 6) -> str:
    """
    Search the knowledge base for information about UFC fights, events, and fighters.
    Use this tool for summaries, "what happened", "how did X beat Y", fight descriptions, and event overviews.

    Args:
        query: The search query
        k: Number of documents to retrieve (default 6)
    """
    docs = vectorstore.similarity_search(query, k=k)

    if not docs:
        return "No relevant documents found."

    results = []
    for i, doc in enumerate(docs, 1):
        results.append(f"Document {i}:\n{doc.page_content}")

    return "\n\n---\n\n".join(results)

print("Tool created: retrieve_documents")
print(retrieve_documents.name)
print(retrieve_documents.description)

Tool created: retrieve_documents
retrieve_documents
Search the knowledge base for information about UFC fights, events, and fighters.
Use this tool for summaries, "what happened", "how did X beat Y", fight descriptions, and event overviews.

Args:
    query: The search query
    k: Number of documents to retrieve (default 6)


In [9]:
@tool
def run_sql_query(sql: str) -> str:
    """
    Execute a SQL query against the UFC database.
    Use this tool for statistical questions, rankings, streaks, "most", "highest", "top", "last N fights", averages, defense stats, etc.

    ======================
    AVAILABLE TABLES
    ======================
    - fighters (fighter_id, name, dob, height_cm, reach_cm)
    - events (event_id, name, date, location)
    - fights (fight_id, event_id, weight_class, method, round, time, bonus, bout_type)
    - fight_performances (fight_id, fighter_id, opponent_id, is_winner,
                          sig_strikes_landed, sig_strikes_attempted,
                          takedowns_landed, takedowns_attempted,
                          knockdowns, submission_attempts,
                          control_time_seconds, fight_minutes, ...)

    ======================
    CRITICAL: OFFENSE vs DEFENSE
    ======================
    Each row in fight_performances is ONE fighter's performance in a fight.

    - fp.takedowns_landed / fp.takedowns_attempted  →  that fighter's OWN offense
    - To get stats AGAINST a fighter (defense), you must look at the OPPONENT's row.

    Example - Takedown Defense for a fighter:
    You need the opponent's takedowns_attempted and takedowns_landed.

    Correct pattern for career takedown defense:

    SELECT
        f.name,
        SUM(opp.takedowns_attempted) AS td_attempted_against,
        SUM(opp.takedowns_landed) AS td_landed_against,
        SUM(opp.takedowns_attempted) - SUM(opp.takedowns_landed) AS td_defended
    FROM fight_performances fp
    JOIN fighters f ON fp.fighter_id = f.fighter_id
    JOIN fight_performances opp
        ON fp.fight_id = opp.fight_id
        AND fp.opponent_id = opp.fighter_id
    WHERE f.name ILIKE '%petr yan%'
    GROUP BY f.name;

    Same logic applies for striking defense, etc.

    ======================
    "LAST N FIGHTS" PATTERN
    ======================
    When asking about a fighter's last 3/5 fights or consecutive streaks,
    rank ALL of their fights (wins + losses), not just wins.

    WITH ranked AS (
        SELECT
            fp.fighter_id,
            fp.is_winner,
            fi.method,
            fi.round,
            e.date,
            ROW_NUMBER() OVER (PARTITION BY fp.fighter_id ORDER BY e.date DESC) as rn
        FROM fight_performances fp
        JOIN fights fi ON fp.fight_id = fi.fight_id
        JOIN events e ON fi.event_id = e.event_id
    )
    SELECT ...
    FROM ranked
    WHERE rn <= 3

    ======================
    OTHER NOTES
    ======================
    - Date lives in the events table, not fights.
    - method values: 'KO/TKO', 'Submission', 'Decision - Unanimous', 'Decision - Split', etc.
    - Only SELECT (or WITH ... SELECT) queries are allowed.
    """
    try:
        cleaned = sql.strip().lower()
        if not (cleaned.startswith("select") or cleaned.startswith("with")):
            return "Error: Only SELECT queries are allowed."

        result = conn.execute(sql).fetchdf()

        if result.empty:
            return "Query returned no results."

        return result.to_string(index=False)

    except Exception as e:
        return f"SQL Error: {str(e)}"

print("Updated run_sql_query tool created")

Updated run_sql_query tool created


In [10]:
@tool
def get_fight(fighter_a: str, fighter_b: str) -> str:
    """
    Get detailed information about a specific fight between two fighters.
    Use this when the user asks "Who won X vs Y?", "What happened in X vs Y?",
    "How did X beat Y?", or any question about a particular matchup.

    Args:
        fighter_a: Name of one fighter (partial names are OK)
        fighter_b: Name of the other fighter (partial names are OK)
    """
    sql = f"""
    SELECT
        e.name AS event,
        e.date,
        e.location,
        fa.name AS winner,
        fb.name AS loser,
        fi.method,
        fi.round,
        fi.time,
        fi.weight_class,
        fi.bonus,
        fp_a.sig_strikes_landed AS winner_sig_strikes,
        fp_b.sig_strikes_landed AS loser_sig_strikes,
        fp_a.knockdowns AS winner_knockdowns,
        fp_b.knockdowns AS loser_knockdowns,
        fp_a.takedowns_landed AS winner_takedowns,
        fp_b.takedowns_landed AS loser_takedowns,
        fp_a.submission_attempts AS winner_sub_attempts,
        fp_b.submission_attempts AS loser_sub_attempts
    FROM fights fi
    JOIN events e ON fi.event_id = e.event_id
    JOIN fight_performances fp_a
        ON fi.fight_id = fp_a.fight_id AND fp_a.is_winner = TRUE
    JOIN fighters fa ON fp_a.fighter_id = fa.fighter_id
    JOIN fight_performances fp_b
        ON fi.fight_id = fp_b.fight_id AND fp_b.is_winner = FALSE
    JOIN fighters fb ON fp_b.fighter_id = fb.fighter_id
    WHERE
        (fa.name ILIKE '%{fighter_a}%' AND fb.name ILIKE '%{fighter_b}%')
        OR
        (fa.name ILIKE '%{fighter_b}%' AND fb.name ILIKE '%{fighter_a}%')
    ORDER BY e.date DESC
    LIMIT 5
    """

    try:
        result = conn.execute(sql).fetchdf()

        if result.empty:
            # Fallback: try searching for each name separately to help debugging
            check_a = conn.execute(f"""
                SELECT name FROM fighters
                WHERE name ILIKE '%{fighter_a}%'
                LIMIT 5
            """).fetchdf()

            check_b = conn.execute(f"""
                SELECT name FROM fighters
                WHERE name ILIKE '%{fighter_b}%'
                LIMIT 5
            """).fetchdf()

            msg = f"No fight found between '{fighter_a}' and '{fighter_b}'.\n"

            if not check_a.empty:
                msg += f"Possible matches for '{fighter_a}': {', '.join(check_a['name'].tolist())}\n"
            else:
                msg += f"No fighter found matching '{fighter_a}'.\n"

            if not check_b.empty:
                msg += f"Possible matches for '{fighter_b}': {', '.join(check_b['name'].tolist())}"
            else:
                msg += f"No fighter found matching '{fighter_b}'."

            return msg

        return result.to_string(index=False)

    except Exception as e:
        return f"Error: {str(e)}"

print("Improved get_fight tool created")

Improved get_fight tool created


In [11]:
@tool
def get_event(event_name: str) -> str:
  """
    Get information about a UFC event, including the fight card and results.
    Use this for questions like "Summarize UFC 300", "What fights were on UFC 294?", or "Who fought at UFC 300?".

    Args:
        event_name: Name of the event (e.g. "UFC 300", "UFC Fight Night")
    """
  sql = f"""
  SELECT
    e.name AS event,
    e.date,
    e.location
    fa.name AS winner,
    fb.name AS loser,
    fi.weight_class,
    fi.method,
    fi.round,
    fi.bonus
  FROM events e
  JOIN fights fi ON fi.event_id = e.event_id
  JOIN fight_performances fp_a on fi.fight_id = fp_a.fight_id AND fp_a.is_winner=True
  JOIN fighters fa ON fp_a.fighter_id = fa.fighter_id
  JOIN fight_performances fp_b on fi.fight_id = fp_b.fight_id AND fp_b.is_winner=False
  JOIN fighters fb ON fp_b.fighter_id = fb.fighter_id
  WHERE e.name ILIKE '%{event_name}%'
  ORDER BY fi.weight_class
    """

  try:
    result = conn.execute(sql).fetchdf()
    if result.empty:
      return f"No event found matching {event_name}"

    return result.to_string(index= False)

  except Exception as e:
    return f"Error: {str(e)}"

print("tool created: get_event")

tool created: get_event


In [12]:
@tool
def get_fighter(fighter_name: str):
  """
    Get a fighter's profile, career record, and recent fights.
    Use this for questions like "Tell me about Islam Makhachev", "Show Jon Jones recent fights", or "What is Alex Pereira's record?".

    Args:
        fighter_name: Name of the fighter
    """
  career_sql = f"""
  SELECT
    f.name,
    COUNT(*) as total_fights,
    SUM(CASE WHEN fp.is_winner THEN 1 ELSE 0 END) AS wins,
    SUM(CASE WHEN NOT fp.is_winner THEN 1 ELSE 0 END) AS losses,
    SUM(fp.sig_strikes_landed) AS total_sig_strikes,
    SUM(fp.takedowns_landed) AS total_takedowns,
    SUM(fp.knockdowns) AS total_knockdowns
  FROM fighters f
  JOIN fight_performances fp on f.fighter_id = fp.fighter_id
  WHERE f.name ILIKE '%{fighter_name}%'
  GROUP BY f.name
  """

  recent_sql = f"""
    SELECT
        e.date,
        e.name AS event,
        CASE WHEN fp.is_winner THEN 'Win' ELSE 'Loss' END AS result,
        opp.name AS opponent,
        fi.method,
        fi.round
    FROM fight_performances fp
    JOIN fighters f ON fp.fighter_id = f.fighter_id
    JOIN fighters opp ON fp.opponent_id = opp.fighter_id
    JOIN fights fi ON fp.fight_id = fi.fight_id
    JOIN events e ON fi.event_id = e.event_id
    WHERE f.name ILIKE '%{fighter_name}%'
    ORDER BY e.date DESC
    LIMIT 5
    """

  try:
        career = conn.execute(career_sql).fetchdf()
        recent = conn.execute(recent_sql).fetchdf()

        if career.empty:
            return f"No fighter found matching '{fighter_name}'."

        output = "=== Career Summary ===\n"
        output += career.to_string(index=False)
        output += "\n\n=== Recent 5 Fights ===\n"
        output += recent.to_string(index=False)

        return output
  except Exception as e:
        return f"Error: {str(e)}"


print("Tool created: get_fighter")

Tool created: get_fighter


In [13]:
tools = [
    retrieve_documents,
    run_sql_query,
    get_fight,
    get_fighter,
    get_event
]
for t in tools:
  print(t)

name='retrieve_documents' description='Search the knowledge base for information about UFC fights, events, and fighters.\nUse this tool for summaries, "what happened", "how did X beat Y", fight descriptions, and event overviews.\n\nArgs:\n    query: The search query\n    k: Number of documents to retrieve (default 6)' args_schema=<class 'langchain_core.utils.pydantic.retrieve_documents'> func=<function retrieve_documents at 0x7ac7414b28e0>
name='run_sql_query' description='Execute a SQL query against the UFC database.\nUse this tool for statistical questions, rankings, streaks, "most", "highest", "top", "last N fights", averages, defense stats, etc.\n\n======================\nAVAILABLE TABLES\n======================\n- fighters (fighter_id, name, dob, height_cm, reach_cm)\n- events (event_id, name, date, location)\n- fights (fight_id, event_id, weight_class, method, round, time, bonus, bout_type)\n- fight_performances (fight_id, fighter_id, opponent_id, is_winner,\n                    

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=2048
)

print(llm.model_name)

openai/gpt-oss-120b


In [15]:
system_prompt = """You are a highly accurate UFC AI Analyst.

STRICT RULES:
1. Only use information that comes from the tools.
2. NEVER invent or assume details that are not present in the tool results.
3. If the tool results do not contain enough information to fully answer, say so clearly.
4. Do not make up methods, scores, head-to-head records, or fight descriptions.
5. Prefer structured data (SQL / get_fight / get_event / get_fighter) for facts (winner, method, round, time, stats).
6. Use retrieve_documents only for additional context, not as a source of hard facts if they conflict with structured data.

When answering:
- Be precise and factual.
- If you are not sure about a detail, omit it rather than guessing.
"""
agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = system_prompt
)

print('agent created')

agent created


In [27]:
def ask(question: str):
    print(f"\nQuestion: {question}")
    print("-" * 60)

    result = agent.invoke({
        "messages": [{"role": "user", "content": question}]
   })



    # Get the final answer
    final_message = result["messages"][-1]
    print(final_message.content)
    return result

# Test 1
ask("How many fights has Khamzat Chimaev won by submission and list those fights")


Question: How many fights has Khamzat Chimaev won by submission and list those fights
------------------------------------------------------------
**Khamzat Chimaev – Submission Wins**

| # | Opponent | Event | Date | Method | Round | Time |
|---|----------|-------|------|--------|-------|------|
| 1 | John Phillips | UFC Fight Night: Kattar vs. Ige | 2020‑07‑15 | Submission | 2 | 1:12 |
| 2 | Li Jingliang | UFC 267: Blachowicz vs. Teixeira | 2021‑10‑30 | Submission | 1 | 3:16 |
| 3 | Kevin Holland | UFC 279: Diaz vs. Ferguson | 2022‑09‑10 | Submission | 1 | 2:13 |
| 4 | Robert Whittaker | UFC 308: Topuria vs. Holloway | 2024‑10‑26 | Submission | 1 | 3:34 |

**Total submission victories:** **4**.


{'messages': [HumanMessage(content='How many fights has Khamzat Chimaev won by submission and list those fights', additional_kwargs={}, response_metadata={}, id='15f540d6-163f-4980-90a9-dd4f253ebc89'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to answer: number of fights Khamzat Chimaev won by submission, and list those fights. Use database. Need to query fights where Khamzat Chimaev is winner and method = 'Submission'. Need fight details: opponent, event, date maybe. Use get_fighter to get his fighter_id? Or use SQL joining fights and fight_performances.\n\nWe need to find fights where fighter is Khamzat Chimaev and is_winner = true and method = 'Submission'.\n\nWe need to list those fights: maybe opponent name, event name, date, method, round, time.\n\nLet's run SQL.\n\n", 'tool_calls': [{'id': 'fc_0d9a0938-7bc0-4f98-9af6-0a6296f02d28', 'function': {'arguments': '{"sql":"WITH chimaev_fights AS (\\n    SELECT fp.fighter_id, fp.is_winner, fi.method, fi.ro

In [ ]:
import gradio as gr

def chat_stream(message, history):
    """
    More robust streaming for create_agent.
    """
    try:
        final_text = ""
        tools_used = []

        for event in agent.stream(
            {"messages": [{"role": "user", "content": message}]},
            stream_mode="values"   # important: use "values"
        ):
            messages = event.get("messages", [])
            if not messages:
                continue

            last = messages[-1]

            # Collect tool names
            tool_calls = getattr(last, "tool_calls", None) or []
            for tc in tool_calls:
                name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", None)
                if name and name not in tools_used:
                    tools_used.append(name)

            # Get text content from AI messages
            content = getattr(last, "content", None)
            if content and isinstance(content, str) and content.strip():
                # Prefer showing tools + final answer
                if tools_used:
                    final_text = "🔧 Tools used: " + ", ".join(f"`{t}`" for t in tools_used) + "\n\n" + content
                else:
                    final_text = content
                yield final_text

        if not final_text:
            # Fallback: non-streaming invoke
            result = agent.invoke({"messages": [{"role": "user", "content": message}]})
            answer = result["messages"][-1].content
            yield answer

    except Exception as e:
        yield f"Error: {str(e)}"


demo = gr.ChatInterface(
    fn=chat_stream,
    title="UFC AI Analyst",
    description="Ask about fights, fighters, events, stats, and rankings.",
    examples=[
        "Who won Pereira vs Hill at UFC 300?",
        "How did Islam Makhachev beat Charles Oliveira?",
        "Who has the most significant strikes since 2020?",
        "Tell me about Alex Pereira",
        "What is Ilia Topuria's knockout rate?",
        "How many knockdowns did Ilia Topuria land against Josh Emmett?"
    ],
    chatbot=gr.Chatbot(height=520),
)

demo.launch(share=True, debug=True)